# GPT-5.6 Prompting Guide Examples

This notebook demonstrates the current (Jul 2026) GPT-5.6 prompting guide using the OpenAI Responses API — **outcome-first, not scaffolding-first**.

The GPT-5.6 prompting guide reversed a lot of earlier (GPT-5 / GPT-5.2-era) advice: GPT-5.6 follows prompt contracts closely enough that heavy XML scaffolding blocks (`<persistence>`, `<context_gathering>`, `<self_reflection>`, `<output_verbosity_spec>`, ...) now mostly get in the model's way instead of helping. Measured impact of trimming that scaffolding: **+10-15% eval scores, -41-66% tokens, -33-67% cost** (source: OpenAI GPT-5.6 prompting guide).

Every section below follows the same **Now / Previously** pattern used in the course deck: lead with current best practice, note the older pattern only so you can recognize and drop it.

## Table of Contents
1. [Setup and Configuration](#setup)
2. [Verbosity Is a Global Parameter](#verbosity)
3. [State the Constraint, Skip the Ceremony](#constraints)
4. [Trust the Default Grounding](#grounding)
5. [Agentic Eagerness — Turning It Down](#eagerness-down)
6. [Agentic Eagerness — Turning It Up](#eagerness-up)
7. [Tool-Calling Preambles](#preambles)
8. [Ask for the Outcome, Not the Self-Review](#outcome)
9. [Reasoning Effort — The Six Levels](#reasoning)
10. [Instruction Following](#instructions)
11. [Markdown Formatting](#markdown)
12. [Metaprompting](#metaprompting)
13. [Production Patterns, Trimmed](#production)
14. [Hands-On Exercise: Trim a Bloated Prompt](#exercise)


> **Model + scope note.** All `model=` calls in this notebook target **`gpt-5.6-sol`** (the bare `gpt-5.6` alias also routes to Sol), current flagship for coding and professional work — it absorbed the dedicated-coding role `gpt-5.3-codex` used to hold. `reasoning.effort` is pinned explicitly in every call; GPT-5.6 defaults to `medium` if you omit it.
>
> This notebook replaces an earlier version built for the original (Aug 2025) GPT-5 prompting guide. That version is kept for reference at `notebooks/legacy/3.0-gpt5-prompting-guide.ipynb` — it's a good side-by-side of exactly what the guide reversed.
>
> Context reuse across turns (`client.conversations.create()` vs. legacy `previous_response_id` chaining) is covered in depth in `notebooks/0.0-introduction-openai-responses-api.ipynb` and isn't repeated here.


## 1. Setup and Configuration <a id='setup'></a>

In [1]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5.6-sol"

print("Setup complete. Ready to demonstrate GPT-5.6 prompting techniques.")


Setup complete. Ready to demonstrate GPT-5.6 prompting techniques.


## 2. Verbosity Is a Global Parameter <a id='verbosity'></a>

**Now:** control output length with the API-level `text.verbosity` parameter (`low`/`medium`/`high`) — a single global switch, not something you write into the prompt. Set once, it applies consistently and doesn't compete with your actual task instructions for the model's attention.


In [2]:
verbosity_levels = ["low", "medium", "high"]
test_query = "Explain how async/await works in JavaScript"

for verbosity in verbosity_levels:
    resp = client.responses.create(
        model=MODEL,
        input=test_query,
        text={"verbosity": verbosity},
        reasoning={"effort": "low"},
    )
    print(f"\n{'='*50}\nVerbosity: {verbosity}")
    print(f"Response length: {len(resp.output_text)} characters")
    print(f"Preview: {resp.output_text[:200]}...")



Verbosity: low
Response length: 1325 characters
Preview: `async/await` is syntax for working with JavaScript Promises in a more readable, synchronous-looking way.

### `async`

Adding `async` to a function makes it always return a Promise:

```js
async func...



Verbosity: medium
Response length: 2565 characters
Preview: `async/await` is JavaScript syntax for working with **Promises** in a way that resembles synchronous code.

## `async`

Adding `async` to a function makes it always return a Promise:

```js
async func...



Verbosity: high
Response length: 9648 characters
Preview: `async`/`await` is JavaScript syntax for working with **Promises** in a way that resembles synchronous code. It makes asynchronous operations—such as network requests, timers, or database calls—easier...


### Mixed Verbosity (Low Global, High for Code)

In [3]:
mixed_verbosity_system = """
Write code for clarity first. Prefer readable, maintainable solutions with clear
names and straightforward control flow. Use high verbosity for code, low verbosity
for everything else.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=mixed_verbosity_system,
    input="Write a function to validate email addresses and explain what it does",
    text={"verbosity": "low"},  # global switch — the instructions above steer code specifically
    reasoning={"effort": "low"},
)
print("Notice: brief explanation, but a fully readable function.")
print(resp.output_text[:800])


Notice: brief explanation, but a fully readable function.
```javascript
/**
 * Performs practical email-address validation.
 *
 * This is intentionally simpler than the full RFC email specification.
 * It checks common formatting rules suitable for most applications.
 */
function isValidEmail(email) {
  if (typeof email !== "string") {
    return false;
  }

  const normalizedEmail = email.trim();

  if (normalizedEmail.length === 0 || normalizedEmail.length > 254) {
    return false;
  }

  const emailPattern =
    /^[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?(?:\.[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?)+$/;

  return emailPattern.test(normalizedEmail);
}
```

The function:

- Rejects non-string, empty, or overly long inputs.
- Requires one `@` separator.
- Validates common characters in the local part.
-


> **Previously:** this needed an in-prompt `<output_verbosity_spec>` block spelling out sentence/bullet counts ("default to 3-6 sentences or <=5 bullets... avoid long narrative paragraphs..."). If that's the mental model you learned, drop it — the guide moved this back to the API layer.
>
> Source: [OpenAI GPT-5.6 prompting guide](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6) · [GPT-5.2 guide](https://developers.openai.com/cookbook/examples/gpt-5/gpt-5-2_prompting_guide) (historical)


## 3. State the Constraint, Skip the Ceremony <a id='constraints'></a>

**Now:** one plain sentence carries the same weight a whole fenced XML block used to need — GPT-5.6 follows prompt contracts closely, so ceremony doesn't add safety margin anymore.


In [4]:
scope_instruction = "Implement only what's requested — no extra features, components, or UX embellishments."

resp = client.responses.create(
    model=MODEL,
    instructions=scope_instruction,
    input="Write a Python function that returns the nth Fibonacci number.",
    reasoning={"effort": "low"},
)
print(resp.output_text)  # Expect: one function, no CLI, no tests, no extras.


```python
def fibonacci(n: int) -> int:
    if n < 0:
        raise ValueError("n must be non-negative")

    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a
```


### Concrete Constraints, Not Absolute Words

**Now:** keep the specific, measurable rules (hard numbers, named defaults) — drop the "always"/"never" framing around them. GPT-5.6 treats a plain rule as a real constraint to satisfy, not a suggestion to weigh against other instructions.


In [5]:
codebase_standards = """
<frontend_stack_defaults>
Framework: Next.js (TypeScript). Styling: Tailwind CSS + shadcn/ui. Icons: Lucide.
</frontend_stack_defaults>

Typography: 4-5 sizes max. Spacing: multiples of 4. Color: 1 neutral base + up to 2 accents.
Loading state: skeleton placeholders. Prefer Radix/shadcn for built-in accessibility.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=codebase_standards,
    input="Add a new user profile card component to the existing dashboard.",
    text={"verbosity": "low"},
    reasoning={"effort": "medium"},
)
print(resp.output_text[:800])


Please share the existing dashboard code or repository files so I can add the user profile card consistently with its current layout and styling.


> **Previously:** the same two rules needed a `<scope_discipline>` XML block plus a `<ui_ux_best_practices>` block full of "Always..." directives. The constraints were always correct — only the packaging (XML tags, absolute-word framing) is what the guide now flags as historical.
>
> Source: [OpenAI GPT-5.6 prompting guide](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6) · [GPT-5 guide — Codebase Design](https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide#matching-codebase-design) (historical)


## 4. Trust the Default Grounding <a id='grounding'></a>

**Now:** re-grounding on long inputs, and flagging unstated assumptions or ungrounded claims, are things GPT-5.6 does reliably **on its own** — you generally don't need to instruct it to outline, re-state constraints, or self-check before answering.

Run the cell below with **no** re-grounding or uncertainty instructions at all, and read the output looking for whether it naturally hedges on the parts it can't know for sure (this varies call to call — that's the point of running it live rather than taking it on faith).


In [6]:
resp = client.responses.create(
    model=MODEL,
    input="Estimate the monthly cost of running a 3-node Postgres cluster on a major cloud.",
    reasoning={"effort": "medium"},
)
print(resp.output_text)


### Rough estimate: **$900–$1,400/month**

For a managed 3-node PostgreSQL cluster on AWS, Azure, or GCP, assuming:

- 1 primary + 2 replicas across availability zones
- 4 vCPU / 16 GB RAM per node
- 500 GB SSD storage
- On-demand pricing, 730 hours/month
- Moderate database I/O
- Same-region application traffic

| Component | Estimated monthly cost |
|---|---:|
| 3 database instances | $700–$950 |
| Replicated SSD storage | $100–$250 |
| Backups, I/O, monitoring | $50–$150 |
| Network traffic | $0–$100 |
| **Total** | **$900–$1,400** |

Examples include AWS RDS PostgreSQL Multi-AZ, Google Cloud SQL with read replicas, or Azure Database for PostgreSQL plus replicas.

### Other configurations

- **Small cluster, 2 vCPU / 8 GB each:** $450–$800/month
- **Larger production cluster, 8 vCPU / 32 GB each:** $1,700–$2,800/month
- **Self-managed on VMs:** approximately $500–$800/month for the baseline configuration, but excludes operational labor and may require extra backup, monitoring, and f

**Reserve explicit instructions for genuinely unusual cases:** adversarial or contradictory source material, or domains where a wrong answer is costly enough that you want the check made visible in the output (see the `compliance_reviewer` personality in [Production Patterns](#production) below for a legitimate use of an explicit caveats instruction).

> **Previously:** this needed explicit step-by-step `<long_context_protocol>` and `<uncertainty_self_check>` blocks written into every long-context or high-risk prompt. GPT-5.6 does this by default — verify occasionally, don't over-prompt for it.
>
> Source: [OpenAI GPT-5.6 prompting guide](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6) (historical: GPT-5.2 guide)


## 5. Agentic Eagerness — Turning It Down <a id='eagerness-down'></a>

**Now:** `reasoning.effort` is still the real lever — set it low (or `none`, GPT-5.6's fastest supported level) for fast, narrow, low-latency tasks. State a stopping condition once, plainly.


In [7]:
search_tools = [
    {
        "type": "function",
        "name": "search_codebase",
        "description": "Search for code patterns in the codebase",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "path": {"type": "string"},
            },
            "required": ["query"],
        },
    },
]

resp = client.responses.create(
    model=MODEL,
    instructions="Search only in src/. Stop once you find the function or after 2 searches.",
    input="Find and fix the authentication bug in the login module",
    tools=search_tools,
    reasoning={"effort": "low"},
)
for item in resp.output:
    if item.type == "message":
        print(item.content[0].text[:600])
    elif item.type == "function_call":
        print(f"[tool call] {item.name}({item.arguments})")


[tool call] search_codebase({"query":"login authentication authenticate password token","path":"src/"})


## 6. Agentic Eagerness — Turning It Up <a id='eagerness-up'></a>

**Now:** raise `reasoning.effort` for tasks that need real persistence through to completion — the model keeps working through a task and verifies edge cases without being told to, so a reminder block isn't doing any work anymore.


In [8]:
resp = client.responses.create(
    model=MODEL,
    instructions="Give a brief one-line update only when starting a new major phase or changing plan.",
    input="Refactor the authentication system to use OAuth 2.0",
    tools=search_tools,
    reasoning={"effort": "high"},  # default is "medium"
)
for item in resp.output:
    if item.type == "message":
        print(item.content[0].text[:600])
    elif item.type == "function_call":
        print(f"[tool call] {item.name}({item.arguments})")


I’ll inspect the existing authentication flow and project structure, then identify the OAuth 2.0 integration points.
[tool call] search_codebase({"query":"authentication auth login logout session token password middleware","path":""})
[tool call] search_codebase({"query":"package.json requirements.txt pyproject.toml go.mod Cargo.toml pom.xml build.gradle","path":""})
[tool call] search_codebase({"query":"routes controllers handlers app server main configuration environment variables","path":""})


> **Previously:** "turning it down" needed a full `<context_gathering>` XML template (early-stop criteria, escalation rules, search-loop policy); "turning it up" needed an explicit `<persistence>` block plus instructions for thorough step-by-step progress narration. The guide names templates like that as trim-worthy on both ends — `reasoning.effort` plus one plain sentence does the same job now.
>
> Source: [OpenAI GPT-5.6 prompting guide](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6) · [GPT-5 guide](https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide) (historical)


## 7. Tool-Calling Preambles <a id='preambles'></a>

**Now:** a brief one-liner when starting a new phase or changing plan is enough — not a running narration of every step. (This is the same idea as [Turning It Up](#eagerness-up) above, applied specifically to preambles — it's broken out here because the guide calls it out as its own recommended pattern.)


In [9]:
file_tools = [
    {
        "type": "function",
        "name": "edit_file",
        "description": "Edit a file in the codebase",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string"}, "content": {"type": "string"}},
            "required": ["path", "content"],
        },
    }
]

resp = client.responses.create(
    model=MODEL,
    instructions="Post one brief line before each new phase of work. No step-by-step narration.",
    input="Create a Python script that reads a CSV of orders and writes a per-region summary.",
    tools=file_tools,
    reasoning={"effort": "low"},
)
for item in resp.output[:4]:
    if item.type == "message":
        print(f"[message] {item.content[0].text[:300]}")
    elif item.type == "function_call":
        print(f"[tool call] {item.name} - {item.arguments[:100]}...")


[message] I’ll add a standalone CLI script that groups orders by region and summarizes order counts and amounts.
[tool call] edit_file - {"path":"summarize_orders.py","content":"#!/usr/bin/env python3\n\"\"\"Read an orders CSV and write ...


## 8. Ask for the Outcome, Not the Self-Review <a id='outcome'></a>

**Now:** state the quality bar once, as an outcome — the model already applies this kind of review internally without being asked to narrate a score.


In [10]:
app_generation_instructions = """
Make sure components are reusable, errors are handled, and the implementation is
reasonably performant before finishing.

Tech stack: Next.js (TypeScript), Tailwind CSS, shadcn/ui, Lucide icons.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=app_generation_instructions,
    input="Create a modern dashboard for analytics with real-time charts",
    reasoning={"effort": "medium"},
)
print(resp.output_text[:600])


Below is a complete responsive analytics dashboard with simulated real-time data, reusable chart components, loading/error states, and pause/resume controls.

Install the chart dependency and required shadcn components:

```bash
npm install recharts
npx shadcn@latest add avatar badge button card dropdown-menu progress separator
```

### `app/dashboard/page.tsx`

```tsx
"use client";

import {
  Component,
  type ErrorInfo,
  type ReactNode,
  useEffect,
  useMemo,
  useState,
} from "react";
import {
  Activity,
  ArrowDownRight,
  ArrowUpRight,
  BarChart3,
  Bell,
  ChevronDown,
  CircleHelp


> **Previously:** this needed a `<self_reflection>` block asking the model to spend time building a 5-7 category rubric, silently score itself 1-10 on each dimension, and iterate until it passed. That scored checklist is exactly the kind of process narration the guide now says to trim.
>
> Source: [OpenAI GPT-5.6 prompting guide](https://developers.openai.com/api/docs/guides/prompt-guidance-gpt-5p6) · [GPT-5 guide — Codebase Design](https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide#matching-codebase-design) (historical)


## 9. Reasoning Effort — The Six Levels <a id='reasoning'></a>

GPT-5.6 tiers (`sol`/`terra`/`luna`) support six `reasoning.effort` values: **`none`, `low`, `medium`, `high`, `xhigh`, `max`**. Default is `medium` if omitted.

> Note: **`minimal`** existed on earlier GPT-5.x releases but was dropped for GPT-5.6 — `none` is now the fastest supported level, not `minimal`. (`notebooks/2.0-gpt5-params.ipynb` currently says `low` is the fastest supported level on GPT-5.6 — that predates this correction and is worth reconciling; see the reasoning-effort research note for sourcing.)


In [11]:
test_message = "Implement a binary search algorithm with proper error handling"

for level in ["none", "low", "medium", "high"]:
    resp = client.responses.create(
        model=MODEL,
        input=test_message,
        reasoning={"effort": level},
    )
    print(f"\nReasoning effort: {level}")
    print(f"Output tokens: {resp.usage.output_tokens}")
    print(f"Preview: {resp.output_text[:150]}...")



Reasoning effort: none
Output tokens: 338
Preview: ```python
def binary_search(items, target):
    """
    Return the index of `target` in a sorted sequence, or -1 if not found.

    Raises:
        Ty...



Reasoning effort: low
Output tokens: 456
Preview: ```python
from collections.abc import Sequence
from typing import TypeVar

T = TypeVar("T")


def binary_search(items: Sequence[T], target: T) -> int:...



Reasoning effort: medium
Output tokens: 930
Preview: ```python
from collections.abc import Sequence
from typing import TypeVar

T = TypeVar("T")


def binary_search(
    items: Sequence[T],
    target: T...



Reasoning effort: high
Output tokens: 1247
Preview: ### Python implementation

```python
from collections.abc import Sequence
from typing import TypeVar

T = TypeVar("T")


def binary_search(
    items:...


## 10. Instruction Following <a id='instructions'></a>

GPT-5.6 follows instructions with surgical precision but needs clear, non-contradictory prompts. Resolve instruction-hierarchy conflicts explicitly rather than leaving them implicit.

In [12]:
clear_instructions = """
You are a code review assistant. Priority order: security (CRITICAL), logic errors
(HIGH), performance (MEDIUM), style (LOW). For each issue: state priority, explain
the problem, suggest a fix. Group by priority, use bullet points, include line
numbers when applicable.
"""

code_to_review = '''
def process_user_input(input_string):
    query = "SELECT * FROM users WHERE name = '" + input_string + "'"  # SQL injection risk
    for i in range(len(result)):
        print(result[i])
    password = "admin123"  # Hardcoded credential
    return result
'''

resp = client.responses.create(
    model=MODEL,
    instructions=clear_instructions,
    input=f"Review this code:\n```python\n{code_to_review}\n```",
    reasoning={"effort": "low"},
)
print(resp.output_text)


## CRITICAL

- **Line 3 — SQL injection vulnerability**
  - User-controlled input is concatenated directly into the SQL query. An attacker could alter the query using input such as `' OR 1=1 --`.
  - **Fix:** Use a parameterized query:
    ```python
    cursor.execute("SELECT * FROM users WHERE name = %s", (input_string,))
    ```
    Use `?` instead of `%s` for database drivers such as `sqlite3`.

- **Line 6 — Hardcoded credential**
  - Embedding passwords in source code risks exposure through version control, logs, or code access. The credential is also unused in this function.
  - **Fix:** Remove it. If a credential is genuinely required, retrieve it from a secrets manager or protected environment variable and rotate the exposed value.

## HIGH

- **Lines 4 and 7 — `result` is undefined**
  - `result` is referenced before assignment, so the function raises `NameError`.
  - **Fix:** Execute the query and fetch the results before iterating:
    ```python
    cursor.execute("SELECT * F

## 11. Markdown Formatting <a id='markdown'></a>

GPT-5.6 doesn't format in Markdown by default in the API but can be prompted to do so. This part of the guide hasn't changed.

In [13]:
markdown_instructions = """
Use Markdown only where semantically correct: `inline code` for functions/variables,
fenced code blocks for code, lists and tables for structured data. Use headers (##)
to organize sections.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=markdown_instructions,
    input="Compare the time complexity of bubble sort, quick sort, and merge sort",
    reasoning={"effort": "low"},
)
print(resp.output_text)


## Time Complexity Comparison

| Algorithm | Best Case | Average Case | Worst Case | Space Complexity |
|---|---:|---:|---:|---:|
| Bubble sort | `O(n)`* | `O(n²)` | `O(n²)` | `O(1)` |
| Quick sort | `O(n log n)` | `O(n log n)` | `O(n²)` | `O(log n)` average** |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n log n)` | `O(n)` |

\* Bubble sort achieves `O(n)` in the best case only when optimized to stop if a pass makes no swaps.  
\** Quick sort uses `O(log n)` recursion space on average and `O(n)` in the worst case.

## Summary

- **Bubble sort:** Simple but inefficient for large inputs due to quadratic average and worst-case time.
- **Quick sort:** Usually very fast in practice, but poor pivot selection can cause `O(n²)` behavior.
- **Merge sort:** Guarantees `O(n log n)` time but requires additional memory for arrays.
- **Stability:** Bubble sort and merge sort are typically stable; standard in-place quick sort is generally not.


## 12. Metaprompting <a id='metaprompting'></a>

Using GPT-5.6 to improve its own prompts — still a solid technique; not something the guide reversed.

In [14]:
metaprompt_template = """
Here's a prompt: {prompt}

The desired behavior is for the agent to {desired}, but instead it {undesired}. While
keeping as much of the existing prompt intact as possible, what minimal edits would
you make to fix this?
"""

metaprompt = metaprompt_template.format(
    prompt="Search for the bug and fix it quickly without asking questions",
    desired="thoroughly investigate the codebase and fix the root cause",
    undesired="makes superficial fixes without proper investigation",
)

resp = client.responses.create(
    model=MODEL,
    input=metaprompt,
    reasoning={"effort": "high"},
)
print(resp.output_text)


**Revised prompt:**

> Thoroughly search the codebase for the root cause of the bug and fix it without asking questions.

This removes “quickly,” which encourages superficial changes, and explicitly prioritizes codebase investigation and root-cause resolution.


## 13. Production Patterns, Trimmed <a id='production'></a>

Real-world prompt configurations (Codex, Cursor, SWE-Bench, retail agent), updated to drop the scaffolding GPT-5.6's guide flags as historical. Compare these against `notebooks/legacy/3.0-gpt5-prompting-guide.ipynb` if you want to see exactly what got cut.


### Codex-Style Agentic Coding

Formerly tied to the dedicated `gpt-5.3-codex` model, now absorbed into `gpt-5.6-sol`. `reasoning.effort: high` replaces the old `<persistence>` block; verification and edit-format rules stay, because they're concrete constraints, not process narration.

In [15]:
codex_instructions = """
Before declaring done: run the build and tests, then read the resulting diff. If
anything fails, fix it and re-verify. Express file changes as patches/diffs against
existing files, not full-file rewrites. One brief line per new phase, nothing more.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=codex_instructions,
    input="Add input validation to the create_user endpoint and update its tests.",
    reasoning={"effort": "high"},  # the real lever — no <persistence> block needed
)
print(resp.output_text[:800])


I’ll inspect the endpoint, validation patterns, and existing tests.I can’t access the repository or run commands in this session. Please provide the `create_user` endpoint, its existing tests, and the project’s build/test commands so I can prepare and verify the patch.


### Prompt Personalities

A reusable persona block that fixes tone, default verbosity, and decision style for a whole product surface. `compliance_reviewer` is the legitimate exception to [Trust the Default Grounding](#grounding): a genuinely high-stakes domain where making the uncertainty check visible in the output is worth the extra instruction.


In [16]:
PERSONALITIES = {
    "terse_senior_engineer": "You are a terse senior engineer. Prefer the direct fix over discussion. Call out the one risk that matters, nothing else.",
    "patient_mentor": "You are a patient mentor. Explain the 'why' before the 'how'. One small analogy, then a concrete example.",
    "compliance_reviewer": "You are a careful compliance reviewer. Flag unstated assumptions and ungrounded claims explicitly. End with a Caveats list.",
}

def ask_as(personality: str, prompt: str, effort: str = "low") -> str:
    return client.responses.create(
        model=MODEL,
        instructions=PERSONALITIES[personality],
        input=prompt,
        text={"verbosity": "low"},
        reasoning={"effort": effort},
    ).output_text

print(ask_as("terse_senior_engineer", "Should we add a Redis cache in front of this query?"))


Not yet. First confirm the query is a measured latency/load bottleneck and optimize/index it. Add Redis only if results are frequently reused and tolerate staleness.

Main risk: cache invalidation causing stale data.


### Cursor-Style Production Prompt

The original Cursor prompt included a `<context_understanding>` block telling the model to gather more info before ending its turn when unsure. Per [Trust the Default Grounding](#grounding), that's now the model's default behavior — dropped below.

In [17]:
cursor_instructions = """
Write code for clarity first: readable names, straightforward control flow, no
code-golf unless asked. Your edits are shown to the user as proposed changes, so
propose the change directly rather than asking whether to proceed.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=cursor_instructions,
    input="Refactor this function to use async/await instead of callbacks",
    text={"verbosity": "low"},
    reasoning={"effort": "medium"},
)
print(resp.output_text[:600])


Please paste the function you want refactored.


### SWE-Bench Verified Configuration

Already close to the current style — concrete verification requirements, not narration. Trimmed slightly (dropped "the user is very patient" filler).

In [18]:
swe_bench_instructions = """
You can run `bash -lc <apply_patch_command>` to apply a diff/patch. Verify changes
against both visible and hidden edge cases before ending your turn — not all tests
are visible in the repository.
"""

apply_patch_tool = {
    "type": "function",
    "name": "apply_patch",
    "description": "Apply a patch to modify files",
    "parameters": {
        "type": "object",
        "properties": {"patch": {"type": "string"}},
        "required": ["patch"],
    },
}

resp = client.responses.create(
    model=MODEL,
    instructions=swe_bench_instructions,
    input="Fix the sorting algorithm to handle edge cases correctly",
    tools=[apply_patch_tool],
    reasoning={"effort": "high"},
)
for item in resp.output:
    if item.type == "message":
        print(item.content[0].text[:500])
    elif item.type == "function_call":
        print(f"[tool call] {item.name}({item.arguments[:150]})")


Please provide the sorting algorithm’s source code or the relevant file path. I currently don’t have access to inspect repository files, so I need that context to identify and fix the edge cases safely.


### Retail Agent Example (Tau-Bench Style)

Already concrete constraints (auth first, confirm before consequential actions, one tool call at a time) — nothing to trim here.

In [19]:
retail_instructions = """
Authenticate the user (email, or name + zip) before helping. Before cancel, modify,
return, or exchange actions, list the action detail and get explicit confirmation.
One tool call at a time. Only act on 'pending' or 'delivered' orders.
"""

retail_tools = [
    {
        "type": "function",
        "name": "authenticate_user",
        "description": "Authenticate user by email or name+zip",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string"},
                "name": {"type": "string"},
                "zip_code": {"type": "string"},
            },
        },
    },
    {
        "type": "function",
        "name": "get_order_status",
        "description": "Get the status of an order",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
]

resp = client.responses.create(
    model=MODEL,
    instructions=retail_instructions,
    input="Hi, I'm John Smith, zip 94103. I want to check on my recent order.",
    tools=retail_tools,
    reasoning={"effort": "low"},
)
for item in resp.output:
    if item.type == "message":
        print(item.content[0].text[:400])
    elif item.type == "function_call":
        print(f"[tool call] {item.name}({item.arguments})")


[tool call] authenticate_user({"email":"","name":"John Smith","zip_code":"94103"})


## Summary and Best Practices

This notebook demonstrated current (Jul 2026) GPT-5.6 prompting technique — outcome-first, not scaffolding-first:

**Trim:**
- XML persistence/context-gathering/self-reflection blocks and repeated style rules that don't change behavior
- Context-gathering templates, self-graded checklists, process narration
- Absolute directives ("always"/"never")

**Keep:**
- Outcome statements, success criteria, stopping conditions
- Hard, concrete constraints (specific numbers, specific formats)
- `text.verbosity` and `reasoning.effort` as the two real API-level levers
- Explicit uncertainty/re-grounding instructions, but only for genuinely high-stakes or adversarial cases

**Measured impact of trimming:** +10-15% eval scores, -41-66% tokens, -33-67% cost (source: OpenAI GPT-5.6 prompting guide).

**Still true from earlier guides:** clear non-contradictory instructions, semantic Markdown, metaprompting to iterate on your own prompts.


## 14. Hands-On Exercise: Trim a Bloated Prompt <a id='exercise'></a>

Below is a deliberately **over-scaffolded** GPT-5.2-era prompt — the kind the old guide recommended. Trim it down to the current GPT-5.6 style: an outcome statement, concrete constraints, and the right API parameters — no XML blocks.

**The over-scaffolded prompt:**

> ```
> <persistence>
> You are an agent - keep going until the task is completely resolved before
> yielding back to the user. Never stop or hand back when you encounter
> uncertainty — decide the most reasonable assumption and proceed.
> </persistence>
>
> <context_gathering>
> Goal: get enough context fast. Parallelize discovery, stop as soon as you can
> act. Start broad, fan out to focused subqueries...
> </context_gathering>
>
> <output_verbosity_spec>
> Default to 3-6 sentences or <=5 bullets. Avoid long narrative paragraphs.
> </output_verbosity_spec>
>
> <scope_discipline>
> Implement EXACTLY and ONLY what is requested. No extra features.
> </scope_discipline>
>
> Task: write a Python function that reads a CSV, drops rows with any null
> values, and returns summary statistics (count, mean, min, max) per numeric
> column as a dict.
> ```

Use the empty cell below to write your improved version, then compare with the reference answer.


In [20]:
# YOUR TURN: rewrite the over-scaffolded prompt above into current GPT-5.6 style.
# Hints: one outcome sentence, one concrete scope constraint, text.verbosity + reasoning.effort
# instead of the XML blocks. Drop the flattery-free but still-redundant process instructions.

improved_instructions = """
TODO: write your trimmed instructions here.
"""

# resp = client.responses.create(
#     model=MODEL,
#     instructions=improved_instructions,
#     input="Here's a sample of my CSV columns: order_id, amount, quantity, region",
#     text={"verbosity": "low"},
#     reasoning={"effort": "low"},
# )
# print(resp.output_text)


<details>
<summary><b>Reference answer (click to reveal)</b></summary>

The GPT-5.6 guide's reversal means the fix here is the opposite of the old exercise: **trim**, not add.

```python
instructions = """
Write a Python function that reads a CSV, drops rows with any null values, and
returns summary statistics (count, mean, min, max) per numeric column as a dict.
Implement only this function — no CLI, no plotting, no extra features.
"""

resp = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input="Here's a sample of my CSV columns: order_id, amount, quantity, region",
    text={"verbosity": "low"},   # replaces <output_verbosity_spec>
    reasoning={"effort": "low"}, # replaces <persistence>/<context_gathering> — a small task, low effort is enough
)
```

**Why it's better:**
- `<persistence>` and `<context_gathering>` are gone — `reasoning.effort` is the real lever, and this is a small, well-scoped task.
- `<output_verbosity_spec>` is gone — `text.verbosity: "low"` does the same job at the API layer.
- `<scope_discipline>` collapsed into one plain sentence ("Implement only this function...") — GPT-5.6 follows it as a real constraint, not a suggestion.
- Four XML blocks and ~15 lines became one paragraph and two API parameters — same behavior, far fewer tokens.

</details>
